# Models Comparison

This notebook reproduces the model-comparison table from CSV results inside `results/hypothesis tests/`.

Outputs:
- Accuracy, recall, precision as mean ± std in percentage
- Parameters column
- CSV and LaTeX exports for the final table

Only CSV result files are used (no weights, no Optuna artifacts).

In [13]:
from pathlib import Path

import numpy as np
import pandas as pd


BASE_DIR = Path('../results/hypothesis tests')
OUT_DIR = BASE_DIR / 'table_replication_outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODELS = [
    {
        'name': 'ShallowConvNet 2017',
        'folder': 'shallowconvnet_10seeds',
        'parameters': '36,522',
    },
    {
        'name': 'EEGNet 2017',
        'folder': 'eegnet_10seeds',
        'parameters': '1,746',
    },
    {
        'name': 'CNN--LSTM 2022',
        'folder': 'cnn_lstm_eegnet_10seeds',
        'parameters': '9,052',
    },
    {
        'name': 'Multi-Stream 2023',
        'folder': 'multistream_10seeds',
        'parameters': '574,082',
    },
    {
        'name': 'IM--CBGT 2024',
        'folder': 'imcbgt_10seeds',
        'parameters': '1,195,266',
    },
    {
        'name': 'T-GARNet 2025',
        'folder': 'tgarnet_10seeds',
        'parameters': '9,071',
    },
    {
        'name': 'EEG-Former 2026 (This work)',
        'folder': 'eegformer_10seeds',
        'parameters': '7,322',
    },
]

In [14]:
def to_percent(values):
    values = pd.to_numeric(pd.Series(values), errors='coerce').to_numpy(dtype=float)
    if np.nanmax(values) <= 1.5:
        return values * 100.0
    return values


def infer_fold_prevalence(reference_df):
    """Infer positive-class prevalence p per fold from rows with recall available."""
    needed = {'fold', 'accuracy', 'balanced_acc', 'recall'}
    if not needed.issubset(reference_df.columns):
        raise ValueError('Reference dataframe needs fold, accuracy, balanced_acc, recall columns.')

    prevalence = {}
    for fold, g in reference_df.groupby('fold'):
        acc = pd.to_numeric(g['accuracy'], errors='coerce').to_numpy(dtype=float)
        bal = pd.to_numeric(g['balanced_acc'], errors='coerce').to_numpy(dtype=float)
        rec = pd.to_numeric(g['recall'], errors='coerce').to_numpy(dtype=float)

        spec = 2.0 * bal - rec
        den = rec - spec
        valid = np.abs(den) > 1e-12
        p_vals = (acc[valid] - spec[valid]) / den[valid]

        prevalence[int(fold)] = float(np.nanmean(p_vals))

    return prevalence


def estimate_recall_precision(row, fold_prevalence):
    a = float(row['accuracy'])
    b = float(row['balanced_acc'])
    f1 = float(row['f1'])
    p = float(fold_prevalence[int(row['fold'])])

    den = 2.0 * p - 1.0
    if abs(den) <= 1e-12:
        return np.nan, np.nan

    recall = (a - 2.0 * (1.0 - p) * b) / den
    recall = float(np.clip(recall, 0.0, 1.0))

    den_p = 2.0 * recall - f1
    if abs(den_p) <= 1e-12:
        return recall, np.nan

    precision = (f1 * recall) / den_p
    precision = float(np.clip(precision, 0.0, 1.0))

    return recall, precision


def format_mean_std(mean_val, std_val):
    if pd.isna(mean_val) or pd.isna(std_val):
        return ''
    return f'{mean_val:.1f} +/- {std_val:.1f}'

In [15]:
# Use CNN-LSTM (which includes explicit recall) to infer fold prevalence.
ref_path = BASE_DIR / 'cnn_lstm_eegnet_10seeds' / 'all_fold_results.csv'
ref_df = pd.read_csv(ref_path)
fold_prevalence = infer_fold_prevalence(ref_df)

print('Inferred fold prevalence (positive class):')
for f in sorted(fold_prevalence):
    print(f'  fold {f}: {fold_prevalence[f]:.6f}')

Inferred fold prevalence (positive class):
  fold 0: 0.458333
  fold 1: 0.541667
  fold 2: 0.458333
  fold 3: 0.625000
  fold 4: 0.458333


In [16]:
rows = []
all_debug = []

for cfg in MODELS:
    model_name = cfg['name']
    csv_path = BASE_DIR / cfg['folder'] / 'all_fold_results.csv'

    if not csv_path.exists():
        raise FileNotFoundError(f'Missing CSV: {csv_path}')

    df = pd.read_csv(csv_path).copy()

    for c in ['seed', 'fold', 'accuracy', 'balanced_acc', 'f1']:
        if c not in df.columns:
            raise ValueError(f'{model_name}: missing required column {c} in {csv_path}')

    df['seed'] = pd.to_numeric(df['seed'], errors='coerce').astype(int)
    df['fold'] = pd.to_numeric(df['fold'], errors='coerce').astype(int)

    has_recall_precision = 'recall' in df.columns and 'precision' in df.columns

    if has_recall_precision:
        # Use direct fold-level metrics for models that provide recall/precision.
        eval_df = pd.DataFrame({
            'accuracy': to_percent(df['accuracy']),
            'recall': to_percent(df['recall']),
            'precision': to_percent(df['precision']),
        })

        acc_mean, acc_std = float(eval_df['accuracy'].mean()), float(eval_df['accuracy'].std(ddof=1))
        rec_mean, rec_std = float(eval_df['recall'].mean()), float(eval_df['recall'].std(ddof=1))
        pre_mean, pre_std = float(eval_df['precision'].mean()), float(eval_df['precision'].std(ddof=1))

        aggregation = 'fold-level (50 evaluations)'

    else:
        # Reconstruct recall/precision fold-by-fold, then summarize over seed means.
        est_rec = []
        est_pre = []
        for _, r in df.iterrows():
            rec, pre = estimate_recall_precision(r, fold_prevalence)
            est_rec.append(rec)
            est_pre.append(pre)

        df['recall_est'] = est_rec
        df['precision_est'] = est_pre

        seed_means = df.groupby('seed', as_index=False).agg(
            accuracy=('accuracy', 'mean'),
            recall=('recall_est', 'mean'),
            precision=('precision_est', 'mean'),
        )

        seed_means['accuracy'] = to_percent(seed_means['accuracy'])
        seed_means['recall'] = to_percent(seed_means['recall'])
        seed_means['precision'] = to_percent(seed_means['precision'])

        acc_mean, acc_std = float(seed_means['accuracy'].mean()), float(seed_means['accuracy'].std(ddof=1))
        rec_mean, rec_std = float(seed_means['recall'].mean()), float(seed_means['recall'].std(ddof=1))
        pre_mean, pre_std = float(seed_means['precision'].mean()), float(seed_means['precision'].std(ddof=1))

        aggregation = 'seed-level means (10 seeds, reconstructed recall/precision)'

    rows.append({
        'Model': model_name,
        'Accuracy_mean_%': acc_mean,
        'Accuracy_std_%': acc_std,
        'Recall_mean_%': rec_mean,
        'Recall_std_%': rec_std,
        'Precision_mean_%': pre_mean,
        'Precision_std_%': pre_std,
        'Parameters': cfg['parameters'],
        'Aggregation': aggregation,
    })

    all_debug.append({
        'Model': model_name,
        'Rows_in_all_fold_results': len(df),
        'Has_recall_precision_columns': has_recall_precision,
        'Aggregation': aggregation,
    })

comparison = pd.DataFrame(rows)
debug_info = pd.DataFrame(all_debug)

comparison['Accuracy (%)'] = comparison.apply(lambda r: format_mean_std(r['Accuracy_mean_%'], r['Accuracy_std_%']), axis=1)
comparison['Recall (%)'] = comparison.apply(lambda r: format_mean_std(r['Recall_mean_%'], r['Recall_std_%']), axis=1)
comparison['Precision (%)'] = comparison.apply(lambda r: format_mean_std(r['Precision_mean_%'], r['Precision_std_%']), axis=1)

final_table = comparison[['Model', 'Accuracy (%)', 'Recall (%)', 'Precision (%)', 'Parameters']]

display(debug_info)
display(final_table)

,Model,Rows_in_all_fold_results,Has_recall_precision_columns,Aggregation
0,ShallowConvNet 2017,50,False,"seed-level means (10 seeds, reconstructed reca..."
1,EEGNet 2017,50,False,"seed-level means (10 seeds, reconstructed reca..."
2,CNN--LSTM 2022,50,True,fold-level (50 evaluations)
3,Multi-Stream 2023,50,False,"seed-level means (10 seeds, reconstructed reca..."
4,IM--CBGT 2024,50,True,fold-level (50 evaluations)
5,T-GARNet 2025,50,False,"seed-level means (10 seeds, reconstructed reca..."
6,EEG-Former 2026 (This work),50,False,"seed-level means (10 seeds, reconstructed reca..."


,Model,Accuracy (%),Recall (%),Precision (%),Parameters
0,ShallowConvNet 2017,78.7 +/- 3.2,84.9 +/- 12.2,78.9 +/- 5.3,"36,522"
1,EEGNet 2017,80.7 +/- 1.3,88.8 +/- 2.0,77.7 +/- 1.2,"1,746"
2,CNN--LSTM 2022,84.2 +/- 7.4,93.8 +/- 8.9,79.9 +/- 7.7,"9,052"
3,Multi-Stream 2023,69.2 +/- 2.9,79.3 +/- 3.6,67.9 +/- 3.3,"574,082"
4,IM--CBGT 2024,70.6 +/- 9.6,80.0 +/- 12.0,68.9 +/- 9.6,"1,195,266"
5,T-GARNet 2025,74.8 +/- 2.9,96.2 +/- 1.3,68.3 +/- 2.1,"9,071"
6,EEG-Former 2026 (This work),84.2 +/- 1.8,92.8 +/- 2.6,80.0 +/- 1.6,"7,322"


In [17]:
# Data-driven consistency checks (no hardcoded target values).
check = comparison[['Model', 'Aggregation', 'Accuracy_mean_%', 'Recall_mean_%', 'Precision_mean_%']].copy()
check['has_all_metrics'] = check[['Accuracy_mean_%', 'Recall_mean_%', 'Precision_mean_%']].notna().all(axis=1)

display(check)

if not check['has_all_metrics'].all():
    raise ValueError('Some models have missing computed metrics. Please inspect CSV inputs.')

print('All models have data-driven Accuracy/Recall/Precision metrics computed from CSV files.')

,Model,Aggregation,Accuracy_mean_%,Recall_mean_%,Precision_mean_%,has_all_metrics
0,ShallowConvNet 2017,"seed-level means (10 seeds, reconstructed reca...",78.666667,84.896970,78.850445,True
1,EEGNet 2017,"seed-level means (10 seeds, reconstructed reca...",80.666667,88.806527,77.725683,True
2,CNN--LSTM 2022,fold-level (50 evaluations),84.250000,93.764103,79.884519,True
3,Multi-Stream 2023,"seed-level means (10 seeds, reconstructed reca...",69.250000,79.252214,67.889046,True
4,IM--CBGT 2024,fold-level (50 evaluations),70.583333,79.990676,68.923438,True
5,T-GARNet 2025,"seed-level means (10 seeds, reconstructed reca...",74.750000,96.198601,68.327847,True
6,EEG-Former 2026 (This work),"seed-level means (10 seeds, reconstructed reca...",84.166667,92.782284,80.036081,True


All models have data-driven Accuracy/Recall/Precision metrics computed from CSV files.


In [18]:
# Save outputs.
full_numeric_path = OUT_DIR / 'models_comparison_numeric.csv'
final_table_path = OUT_DIR / 'models_comparison_table.csv'
latex_path = OUT_DIR / 'models_comparison_table.tex'

comparison.to_csv(full_numeric_path, index=False)
final_table.to_csv(final_table_path, index=False)

latex_df = final_table.copy()
latex = latex_df.to_latex(index=False, escape=False)
latex_path.write_text(latex, encoding='utf-8')

print('Saved files:')
print(full_numeric_path)
print(final_table_path)
print(latex_path)

Saved files:
..\results\hypothesis tests\table_replication_outputs\models_comparison_numeric.csv
..\results\hypothesis tests\table_replication_outputs\models_comparison_table.csv
..\results\hypothesis tests\table_replication_outputs\models_comparison_table.tex
